2024 and 2015 NSDUH Dataset Analysis <br>
Xavier McFadden | Data Science for Addiction Research Fellowship | Spring 2026 <br>
Exploratory analysis of stimulant and sedative misuse in the past year between ages 16 and 25 using NSDUH datasets (2024 and 2015 respectively) <br>
Due 3/6/26

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.join(os.getcwd(),'..', 'scripts'))
from load_data import load_2024, load_2015, load_2024_full, load_all_years# type: ignore
from analyze import health_breakdown, distress_correlation, demographic_breakdown, age_group_comparison, misuse_counts, get_distress_pct, get_health_pcts, get_income_pcts
from visualize import plot_trend_line, plot_race_rates, plot_distress_age_comparison, plot_health_breakdown, plot_income_breakdown, plot_misuse_counts, plot_population_income, plot_distress_with_nonusers

output_path = os.path.join(os.getcwd(), '..', 'output')
data_dir = os.path.join(os.getcwd(), '..', 'data')

In [2]:
df_filtered2024 = load_2024(os.path.join(os.getcwd(),'..', 'data', 'NSDUH_2024_Tab.txt'))
df_filtered2015 = load_2015(os.path.join(os.getcwd(),'..', 'data', 'NSDUH_2015_Tab.tsv'))
df_full_2024 = load_2024_full(os.path.join(os.getcwd(),'..', 'data', 'NSDUH_2024_Tab.txt'))
print(df_filtered2024.shape)
print(df_filtered2015.shape)

(18011, 13)
(19131, 11)


In [3]:
datasets = load_all_years(os.path.join(os.getcwd(), '..', 'data'))

2015: (19131, 11)
2016: (18369, 11)
2018: (18181, 11)
2020: (9714, 11)
2022: (18066, 11)
2024: (18011, 13)


In [4]:
stim =[]
sedt =[]
years = []
# misuse counts for trend line
for year, df in datasets.items():
    counts = misuse_counts(df, ['STMNMREC', 'SEDNMREC'])
    stim.append(int(counts['STMNMREC']))
    sedt.append(int(counts['SEDNMREC']))
    years.append(year)
    print(f"{year} — Stimulants: {int(counts['STMNMREC'])}, Sedatives: {int(counts['SEDNMREC'])}")


series = {"Stimulants" :stim, "Sedatives": sedt}

2015 — Stimulants: 1219, Sedatives: 145
2016 — Stimulants: 1144, Sedatives: 125
2018 — Stimulants: 965, Sedatives: 92
2020 — Stimulants: 406, Sedatives: 52
2022 — Stimulants: 572, Sedatives: 57
2024 — Stimulants: 408, Sedatives: 45


In [5]:
# race rates for graph
total_2024 = datasets[2024]['NEWRACE2'].value_counts().sort_index()
misusers_2024 = datasets[2024][datasets[2024]['STMNMREC'].isin([1,2])]
stim_rates = (misusers_2024['NEWRACE2'].value_counts().sort_index() / total_2024 * 100).round(1)
print(stim_rates)

NEWRACE2
1    3.0
2    0.8
3    2.0
4    1.9
5    1.4
6    3.2
7    1.7
Name: count, dtype: float64


In [6]:
# stress correlation
print(distress_correlation(df_filtered2024, ['STMNMREC', 'SEDNMREC'], 'SPDPSTYR', ))
print(distress_correlation(df_filtered2015, ['STMNMREC', 'SEDNMREC'], 'SPDYR'))

#health spread for both datasets

print(health_breakdown(df_filtered2024, ['STMNMREC', 'SEDNMREC']))
print(health_breakdown(df_filtered2015, ['STMNMREC', 'SEDNMREC']))

{'STMNMREC': {'distressed': np.int64(199), 'not_distressed': np.int64(152)}, 'SEDNMREC': {'distressed': np.int64(31), 'not_distressed': np.int64(8)}}
{'STMNMREC': {'distressed': np.int64(337), 'not_distressed': np.int64(695)}, 'SEDNMREC': {'distressed': np.int64(50), 'not_distressed': np.int64(63)}}
{'STMNMREC': {1: 57, 2: 163, 3: 131, 4: 50, 5: 7}, 'SEDNMREC': {1: 1, 2: 14, 3: 13, 4: 13, 5: 4}}
{'STMNMREC': {1: 315, 2: 537, 3: 289, 4: 73, 5: 5}, 'SEDNMREC': {1: 31, 2: 71, 3: 37, 4: 6}}


In [7]:
# demographic breakdown of stimulant and sedative in both datasets.
print(demographic_breakdown(df_filtered2024, ['STMNMREC', 'SEDNMREC'], ['IRSEX', 'NEWRACE2', 'IREDUHIGHST2', 'IRFAMIN3']))
print(demographic_breakdown(df_filtered2015, ['STMNMREC', 'SEDNMREC'], ['IRSEX', 'NEWRACE2', 'IREDUHIGHST2', 'IRFAMIN3']))

{'STMNMREC': {'IRSEX': {1: 205, 2: 203}, 'NEWRACE2': {1: 253, 7: 77, 6: 36, 2: 21, 5: 12, 3: 7, 4: 2}, 'IREDUHIGHST2': {8: 112, 9: 110, 11: 72, 7: 46, 10: 26, 6: 26, 5: 11, 1: 3, 4: 1, 3: 1}, 'IRFAMIN3': {7: 156, 6: 58, 5: 48, 2: 46, 1: 44, 4: 35, 3: 21}}, 'SEDNMREC': {'IRSEX': {1: 23, 2: 22}, 'NEWRACE2': {1: 27, 7: 11, 6: 3, 2: 3, 5: 1}, 'IREDUHIGHST2': {8: 11, 9: 9, 7: 7, 11: 7, 6: 7, 10: 3, 3: 1}, 'IRFAMIN3': {1: 12, 7: 11, 6: 5, 3: 5, 5: 4, 4: 4, 2: 4}}}
{'STMNMREC': {'IRSEX': {1: 649, 2: 570}, 'NEWRACE2': {1: 871, 7: 176, 6: 68, 2: 62, 5: 27, 3: 10, 4: 5}, 'IREDUHIGHST2': {9: 436, 8: 249, 11: 183, 7: 128, 6: 93, 10: 83, 5: 35, 4: 5, 1: 3, 3: 2, 2: 2}, 'IRFAMIN3': {7: 362, 1: 233, 2: 157, 6: 143, 3: 121, 4: 115, 5: 88}}, 'SEDNMREC': {'IRSEX': {2: 79, 1: 66}, 'NEWRACE2': {1: 97, 7: 22, 6: 13, 2: 7, 5: 4, 3: 1, 4: 1}, 'IREDUHIGHST2': {9: 49, 8: 32, 7: 22, 11: 13, 6: 12, 5: 10, 10: 6, 4: 1}, 'IRFAMIN3': {7: 53, 4: 18, 2: 17, 3: 17, 1: 16, 6: 13, 5: 11}}}


In [8]:
# age group comparison (26-35)
print(age_group_comparison(df_full_2024, ['STMNMREC', 'SEDNMREC'], 'AGE3', (7, 8)))

# distress among older age group
df_2024older = df_full_2024[df_full_2024['AGE3'].between(7, 8)]
print(df_2024older[df_2024older['STMNMREC'].isin([1,2])]['SPDPSTYR'].value_counts())
print(df_2024older[df_2024older['SEDNMREC'].isin([1,2])]['SPDPSTYR'].value_counts())

{'STMNMREC': np.int64(253), 'SEDNMREC': np.int64(32)}
SPDPSTYR
0.0    143
1.0    110
Name: count, dtype: int64
SPDPSTYR
0.0    21
1.0    11
Name: count, dtype: int64


In [9]:
# visualize misuse trend line
years = [2015, 2016, 2018, 2020, 2022, 2024]
stim_pcts = []
sed_pcts = []
# calculate percentages for each year
for year in years:
    df = datasets[year]
    total = len(df)
    counts = misuse_counts(df, ['STMNMREC', 'SEDNMREC'])
    stim_pcts.append(round(counts['STMNMREC'] / total * 100, 2)) 
    sed_pcts.append(round(counts['SEDNMREC'] / total * 100, 2))

plot_trend_line(output_path, years, {'Stimulants': stim_pcts, 'Sedatives': sed_pcts}, figure_num=1, caption='Prescription stimulant and sedative misuse rates (% of sample) among adults aged 16-25 across six NSDUH survey years (2015-2024).')

In [10]:
stim =[]
sedt =[]
years = []
# misuse counts for trend line
for year, df in datasets.items():
    counts = misuse_counts(df, ['STMNMREC', 'SEDNMREC'])
    stim.append(int(counts['STMNMREC']))
    sedt.append(int(counts['SEDNMREC']))
    years.append(year)
plot_misuse_counts(output_path, years, {'Stimulants': stim, 'Sedatives': sedt}, figure_num=2, caption='Total number of past-year prescription drug misusers aged 16-25 across six NSDUH survey years (2015-2024).')

In [11]:
# visualize health breakdown
stim_2015_health = get_health_pcts(datasets[2015], 'STMNMREC').tolist()
stim_2024_health = get_health_pcts(datasets[2024], 'STMNMREC').tolist()
sed_2015_health = get_health_pcts(datasets[2015], 'SEDNMREC').tolist()
sed_2024_health = get_health_pcts(datasets[2024], 'SEDNMREC').tolist()

plot_health_breakdown(output_path, stim_2015_health, stim_2024_health, sed_2015_health, sed_2024_health, figure_num=3, caption='Self-reported health status distribution among past-year misusers in 2015 and 2024, by drug category.')

In [12]:
# visualize income breakdown
income_labels = ['<$10k', '$10-19k', '$20-29k', '$30-39k', '$40-49k', '$50-74k', '$75k+']

stim_2015_inc = get_income_pcts(datasets[2015], 'STMNMREC')
stim_2024_inc = get_income_pcts(datasets[2024], 'STMNMREC')
sed_2015_inc = get_income_pcts(datasets[2015], 'SEDNMREC')
sed_2024_inc = get_income_pcts(datasets[2024], 'SEDNMREC')

plot_income_breakdown(output_path, stim_2015_inc, stim_2024_inc, sed_2015_inc, sed_2024_inc, income_labels, figure_num=4, caption='Household income distribution among past-year prescription drug misusers in 2015 and 2024.')

In [13]:
income_labels = ['<$10k', '$10-19k', '$20-29k', '$30-39k', '$40-49k', '$50-74k', '$75k+']
pop_2015 = [14.2, 13.8, 12.2, 11.4, 10.5, 13.1, 24.7]
pop_2024 = [10.4, 10.5, 9.2, 9.2, 10.5, 15.2, 34.9]
plot_population_income(output_path, pop_2015, pop_2024, income_labels, figure_num=5, caption='Household income distribution of the general sample population aged 16-25 in 2015 and 2024.')

In [20]:
older = df_full_2024[df_full_2024['AGE3'].between(7, 8)]
stim_older = older[older['STMNMREC'].isin([1,2])].dropna(subset=['SPDPSTYR'])
sed_older = older[older['SEDNMREC'].isin([1,2])].dropna(subset=['SPDPSTYR'])
stim_age = [get_distress_pct(datasets[2015], 'STMNMREC', 'SPDYR'), get_distress_pct(datasets[2024], 'STMNMREC', 'SPDPSTYR'), round(stim_older['SPDPSTYR'].eq(1).sum() / len(stim_older) * 100, 1)]
sed_age = [get_distress_pct(datasets[2015], 'SEDNMREC', 'SPDYR'), get_distress_pct(datasets[2024], 'SEDNMREC', 'SPDPSTYR'), round(sed_older['SPDPSTYR'].eq(1).sum() / len(sed_older) * 100, 1)]
age_labels = ['2015\n(16-25)', '2024\n(16-25)', '2024\n(26-35)']

plot_distress_age_comparison(output_path, stim_age, sed_age, age_labels, figure_num=7, caption='Psychological distress rates among past-year misusers across three groups: 2015 ages 16-25, 2024 ages 16-25, and 2024 ages 26-35.')

In [21]:
# calculate distress rates
stim_2015_dist = get_distress_pct(datasets[2015], 'STMNMREC', 'SPDYR')
stim_2024_dist = get_distress_pct(datasets[2024], 'STMNMREC', 'SPDPSTYR')
sed_2015_dist = get_distress_pct(datasets[2015], 'SEDNMREC', 'SPDYR')
sed_2024_dist = get_distress_pct(datasets[2024], 'SEDNMREC', 'SPDPSTYR')

non_mis_2015 = datasets[2015][~(datasets[2015]['STMNMREC'].isin([1,2]) | datasets[2015]['SEDNMREC'].isin([1,2]))].dropna(subset=['SPDYR'])
non_mis_2024 = datasets[2024][~(datasets[2024]['STMNMREC'].isin([1,2]) | datasets[2024]['SEDNMREC'].isin([1,2]))].dropna(subset=['SPDPSTYR'])
non_mis_2015_dist = round((non_mis_2015['SPDYR'] == 1).sum() / len(non_mis_2015) * 100, 1)
non_mis_2024_dist = round((non_mis_2024['SPDPSTYR'] == 1).sum() / len(non_mis_2024) * 100, 1)

plot_distress_with_nonusers(output_path, stim_2015_dist, stim_2024_dist, sed_2015_dist, sed_2024_dist, non_mis_2015_dist, non_mis_2024_dist, figure_num=6, caption='Psychological distress rates among stimulant misusers, sedative misusers, and non-misusers in 2015 and 2024.')

In [22]:
# visualize race rates
groups = ['White', 'Black', 'Native Am.', 'Pacific Isl.', 'Asian', 'Multiracial', 'Hispanic']

total_2015 = datasets[2015]['NEWRACE2'].value_counts().sort_index()
total_2024 = datasets[2024]['NEWRACE2'].value_counts().sort_index()

stim_2015_race = (datasets[2015][datasets[2015]['STMNMREC'].isin([1,2])]['NEWRACE2'].value_counts().sort_index() / total_2015 * 100).round(1).tolist()
stim_2024_race = (datasets[2024][datasets[2024]['STMNMREC'].isin([1,2])]['NEWRACE2'].value_counts().sort_index() / total_2024 * 100).round(1).tolist()
sed_2015_race = (datasets[2015][datasets[2015]['SEDNMREC'].isin([1,2])]['NEWRACE2'].value_counts().sort_index() / total_2015 * 100).round(1).fillna(0).tolist()
sed_2024_race = (datasets[2024][datasets[2024]['SEDNMREC'].isin([1,2])]['NEWRACE2'].value_counts().sort_index() / total_2024 * 100).round(1).fillna(0).tolist()

plot_race_rates(output_path, groups, stim_2015_race, stim_2024_race, sed_2015_race, sed_2024_race, figure_num=8, caption='Prescription drug misuse rates within each racial and ethnic group in 2015 and 2024, calculated as percentage of each group reporting past-year misuse.')

In [17]:
#finding health breakdown among non-misusers for comparison

non_stim_2015 = datasets[2015][~datasets[2015]['STMNMREC'].isin([1,2])]
total = len(non_stim_2015)
health_counts = non_stim_2015['HEALTH'].value_counts().reindex([1,2,3,4,5], fill_value=0)
print((health_counts / total * 100).round(1))

non_stim = datasets[2024][~datasets[2024]['STMNMREC'].isin([1,2])]
total = len(non_stim)
health_counts = non_stim['HEALTH'].value_counts().reindex([1,2,3,4,5], fill_value=0)
print((health_counts / total * 100).round(1))

HEALTH
1    29.3
2    39.8
3    24.3
4     6.1
5     0.5
Name: count, dtype: float64
HEALTH
1    25.7
2    35.9
3    28.7
4     8.6
5     1.0
Name: count, dtype: float64


In [18]:
# income distribution for full sample 2015
total_2015 = len(datasets[2015])
income_2015 = datasets[2015]['IRFAMIN3'].value_counts().reindex([1,2,3,4,5,6,7], fill_value=0)
print((income_2015 / total_2015 * 100).round(1))

# income distribution for full sample 2024
total_2024 = len(datasets[2024])
income_2024 = datasets[2024]['IRFAMIN3'].value_counts().reindex([1,2,3,4,5,6,7], fill_value=0)
print((income_2024 / total_2024 * 100).round(1))

IRFAMIN3
1    14.2
2    13.8
3    12.2
4    11.4
5    10.5
6    13.1
7    24.7
Name: count, dtype: float64
IRFAMIN3
1    10.4
2    10.5
3     9.2
4     9.2
5    10.5
6    15.2
7    34.9
Name: count, dtype: float64


In [19]:
# non-misuser distress 2024
non_misusers_2024 = datasets[2024][~datasets[2024]['STMNMREC'].isin([1,2])].dropna(subset=['SPDPSTYR'])
print((non_misusers_2024['SPDPSTYR'].value_counts() / len(non_misusers_2024) * 100).round(1))

# non-misuser distress 2015
non_misusers_2015 = datasets[2015][~datasets[2015]['STMNMREC'].isin([1,2])].dropna(subset=['SPDYR'])
print((non_misusers_2015['SPDYR'].value_counts() / len(non_misusers_2015) * 100).round(1))

SPDPSTYR
0.0    71.5
1.0    28.5
Name: count, dtype: float64
SPDYR
0.0    80.4
1.0    19.6
Name: count, dtype: float64
